# Waste Segregation Classifier — Google Colab

`Input image → Preprocessing (resize / normalize / enhance) → CNN classifier → Predicted bin`

**Classes:** `wet`, `dry`, `metal`
**Data:** [Garbage Classification (12 classes)](https://www.kaggle.com/datasets/mostafaabla/garbage-classification) — no Kaggle login needed.
**Model:** EfficientNetV2B0, ImageNet weights, two-phase transfer learning (same recipe as the skin-disease notebook).

---

### How the 12 source categories become 3 bins

| Bin | Built from | Notes |
|---|---|---|
| `wet` | `biological` | food scraps, peels, organic waste |
| `dry` | `paper`, `cardboard`, `plastic`, `brown-glass`, `green-glass`, `white-glass`, `clothes`, `shoes`, `trash` | the usual municipal "dry waste" bucket |
| `metal` | `metal` | cans, tins, foil, metal scrap |
| *(excluded)* | `battery` | hazardous waste — it belongs in none of the three bins, and forcing it into `dry` would teach the model something unsafe |

Three things are handled on purpose, and worth saying in your write-up:

- **Class imbalance.** `clothes` and `shoes` alone are ~7,000 of ~15,500 images and `wet` / `metal`
  are under 1,000 each. Every source folder is capped (`max_per_folder`) so `dry` is a mix of
  materials rather than "mostly clothes", and class weights cover the remaining gap.
- **No official split.** This dataset ships as one folder, so the notebook makes its own
  stratified **train / val / test** split (70 / 15 / 15). Exact duplicate files are removed
  *before* splitting so the same image can't land in both train and test.
- **The test set is only a loose guide to real-world accuracy.** These are mostly clean
  single-item photos on plain backgrounds. Near-duplicates (same item, slightly different crop)
  can still straddle the split, and a phone photo of a cluttered bin will be harder. Expect
  the deployed number to be lower than the notebook's — test with your own photos before
  trusting it.

## Step 0 — Environment check

In [ ]:
import sys, subprocess, platform
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "gradio"],
               check=False)

import tensorflow as tf
import numpy as np, sklearn

print("Python      :", platform.python_version())
print("TensorFlow  :", tf.__version__)
print("Keras       :", tf.keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    details = tf.config.experimental.get_device_details(gpus[0])
    print("\nGPU         :", details.get("device_name", gpus[0].name))
    cc = details.get("compute_capability")
    print("Compute cap :", cc, "-- float16 tensor cores" if cc and cc >= (7, 0) else "")
else:
    print("\nNO GPU. Runtime -> Change runtime type -> T4 GPU.")

## Step 1 — Configuration

`SOURCE_FOLDERS` is the whole class definition. Move a folder between bins (or into
`DROPPED_FOLDERS`) and re-run from here to see how the confusion matrix reacts.

In [ ]:
from pathlib import Path

CONFIG = {
    "backbone":        "efficientnetv2b0",   # see BACKBONES
    "img_size":        224,
    "batch_size":      64,
    "mixed_precision": True,      # float16 tensor cores: ~1.5-2x faster on T4
    "seed":            1337,

    "split":           (0.70, 0.15, 0.15),   # train / val / test
    "max_per_folder":  1000,       # cap per SOURCE folder -- keeps `dry` from being mostly clothes
                                  # (None = use everything; try 300 for a fast smoke run)

    "epochs_head":     6,         # frozen backbone plateaus fast
    "epochs_finetune": 20,
    "lr_head":         1e-3,
    "lr_finetune":     1e-4,      # cosine-decayed
    "unfreeze_frac":   0.35,      # freeze the bottom 35% of the backbone
    "dropout":         0.35,
    "shuffle_buffer":  1500,      # in IMAGES (dataset is unbatched here)

    "work_dir":   Path("/content/waste_data"),
    "cache_dir":  Path("/content/tf_cache"),
    "export_dir": Path("/content/exported_model"),
}

# name -> (constructor, external preprocessing fn or None, native resolution)
BACKBONES = {
    "efficientnetv2b0": (
        lambda shape: tf.keras.applications.EfficientNetV2B0(
            input_shape=shape, include_top=False, weights="imagenet",
            include_preprocessing=True),   # normalization is inside the backbone
        None, 224),
    "efficientnetv2b2": (
        lambda shape: tf.keras.applications.EfficientNetV2B2(
            input_shape=shape, include_top=False, weights="imagenet",
            include_preprocessing=True),
        None, 260),
    "mobilenetv2": (
        lambda shape: tf.keras.applications.MobileNetV2(
            input_shape=shape, include_top=False, weights="imagenet"),
        tf.keras.applications.mobilenet_v2.preprocess_input, 224),
}

# bin -> source folder names (exact match, case-insensitive)
SOURCE_FOLDERS = {
    "wet":   ["biological"],
    "dry":   ["paper", "cardboard", "plastic", "brown-glass", "green-glass",
              "white-glass", "clothes", "shoes", "trash"],
    "metal": ["metal"],
}

# folder -> why it is left out
DROPPED_FOLDERS = {
    "battery": "hazardous waste -- belongs in none of wet / dry / metal",
}

CLASS_NAMES = list(SOURCE_FOLDERS.keys())
NUM_CLASSES = len(CLASS_NAMES)

for d in ("work_dir", "cache_dir", "export_dir"):
    CONFIG[d].mkdir(parents=True, exist_ok=True)

print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"backbone: {CONFIG['backbone']}  |  batch {CONFIG['batch_size']}  "
      f"|  mixed precision {CONFIG['mixed_precision']}")

## Step 2 — Download the dataset

**No Kaggle login.** The dataset is public and `kagglehub` serves public datasets
anonymously; the cell only asks for a token if the anonymous request is refused.

~250 MB, about a minute.

In [ ]:
import os, kagglehub

DATASET = "mostafaabla/garbage-classification"

def download():
    try:
        return Path(kagglehub.dataset_download(DATASET))
    except Exception as e:
        print(f"Anonymous download failed ({type(e).__name__}: {e})")
        print("Falling back to an API token -- kaggle.com/settings/account -> Create New Token")
        kagglehub.login()
        return Path(kagglehub.dataset_download(DATASET))

RAW_PATH = download()
print("Downloaded to:", RAW_PATH)

KNOWN = {n for names in SOURCE_FOLDERS.values() for n in names} | set(DROPPED_FOLDERS)

def find_class_root(root: Path) -> Path:
    """The directory whose children are the class folders (biological, metal, ...)."""
    for c in [root, *[p for p in root.rglob("*") if p.is_dir()]]:
        hits = {p.name.lower() for p in c.iterdir() if p.is_dir()} & KNOWN
        if len(hits) >= 3:
            return c
    raise FileNotFoundError(f"Could not find the class folders under {root}")

CLASS_ROOT = find_class_root(RAW_PATH)
print("Class root   :", CLASS_ROOT)
print("Folders      :", sorted(p.name for p in CLASS_ROOT.iterdir() if p.is_dir()))

### Build the 3-bin dataset

For each source folder: shuffle, cap at `max_per_folder`, drop exact duplicates
(by MD5), then split every bin **stratified** into train / val / test. Files are
symlinked, so nothing is copied. The cell prints what fed each bin and what was
deliberately skipped.

In [ ]:
import shutil, random, hashlib
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def bin_of(folder_name: str):
    low = folder_name.lower()
    for target, names in SOURCE_FOLDERS.items():
        if low in names:
            return target
    return None

rng = random.Random(CONFIG["seed"])
per_bin = {c: [] for c in CLASS_NAMES}       # bin -> [Path]
used    = {c: {} for c in CLASS_NAMES}       # bin -> {source folder: n kept}
skipped = {}                                 # folder -> reason
seen, n_dups = set(), 0

for folder in sorted(p for p in CLASS_ROOT.iterdir() if p.is_dir()):
    target = bin_of(folder.name)
    if target is None:
        skipped[folder.name] = DROPPED_FOLDERS.get(folder.name.lower(), "not in SOURCE_FOLDERS")
        continue
    files = sorted(f for f in folder.iterdir() if f.suffix.lower() in IMG_EXT)
    rng.shuffle(files)
    kept = 0
    for f in files:
        if CONFIG["max_per_folder"] and kept >= CONFIG["max_per_folder"]:
            break
        digest = hashlib.md5(f.read_bytes()).hexdigest()
        if digest in seen:
            n_dups += 1
            continue
        seen.add(digest)
        per_bin[target].append(f)
        kept += 1
    used[target][folder.name] = kept

# stratified split -> symlinks
if CONFIG["work_dir"].exists():
    shutil.rmtree(CONFIG["work_dir"])
SPLITS = ("train", "val", "test")
fr_train, fr_val, _ = CONFIG["split"]
counts = {s: {} for s in SPLITS}

for c in CLASS_NAMES:
    files = per_bin[c][:]
    rng.shuffle(files)
    n = len(files)
    n_train, n_val = int(n * fr_train), int(n * fr_val)
    parts = {"train": files[:n_train],
             "val":   files[n_train:n_train + n_val],
             "test":  files[n_train + n_val:]}
    for s, fs in parts.items():
        out = CONFIG["work_dir"] / s / c
        out.mkdir(parents=True, exist_ok=True)
        for f in fs:
            os.symlink(f.resolve(), out / f"{f.parent.name}__{f.name}")
        counts[s][c] = len(fs)

TRAIN_DIR, VAL_DIR, TEST_DIR = (CONFIG["work_dir"] / s for s in SPLITS)

print("Folders used")
print("-" * 66)
for c in CLASS_NAMES:
    print(f"{c:>6} <- " + ", ".join(f"{k} ({v})" for k, v in used[c].items()))
print("\nDeliberately skipped")
print("-" * 66)
for k, why in skipped.items():
    print(f"{k:>10} xx {why}")
print(f"\nExact duplicates removed: {n_dups}")

print("\nImage counts")
print("-" * 66)
print(f"{'class':>8} {'train':>8} {'val':>8} {'test':>8}")
for c in CLASS_NAMES:
    print(f"{c:>8} {counts['train'][c]:>8} {counts['val'][c]:>8} {counts['test'][c]:>8}")
tot = {s: sum(counts[s].values()) for s in SPLITS}
print(f"{'TOTAL':>8} {tot['train']:>8} {tot['val']:>8} {tot['test']:>8}")

imbalance = max(counts["train"].values()) / max(1, min(counts["train"].values()))
print(f"\nImbalance ratio (largest/smallest): {imbalance:.1f}x -- handled by class weights")

## Step 3 — Input pipeline

Two memory traps are avoided here, both of which can get a Colab runtime killed (12.7 GB RAM):

- **`.cache()` with no argument caches into RAM.** The cache goes to **disk**
  (`/content/tf_cache`) instead, so JPEGs are decoded once and later epochs read decoded
  tensors off the local SSD — which also keeps the GPU from waiting on the CPU.
- **`.shuffle(N)` on an already-batched dataset buffers N *batches*, not N images.**
  Shuffling happens **before** batching, so the buffer is counted in images.

Train / val / test are separate folders (made in Step 2), so no `validation_split` is used.

In [ ]:
import shutil
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

BACKBONE_FN, PREPROCESS, NATIVE_SIZE = BACKBONES[CONFIG["backbone"]]
if CONFIG["img_size"] != NATIVE_SIZE:
    print(f"note: {CONFIG['backbone']} is native at {NATIVE_SIZE}px, "
          f"using {CONFIG['img_size']}px")
IMG_SIZE = (CONFIG["img_size"], CONFIG["img_size"])

# batch_size=None -> individual images, so shuffle counts IMAGES not batches
common = dict(labels="inferred", label_mode="int", class_names=CLASS_NAMES,
              image_size=IMG_SIZE, batch_size=None, seed=CONFIG["seed"])

train_raw = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, shuffle=True,  **common)
val_raw   = tf.keras.utils.image_dataset_from_directory(VAL_DIR,   shuffle=False, **common)
test_raw  = tf.keras.utils.image_dataset_from_directory(TEST_DIR,  shuffle=False, **common)

print("\nLabel order:", train_raw.class_names)
assert train_raw.class_names == CLASS_NAMES

# Stale cache files from an earlier run with different data would be silently wrong.
shutil.rmtree(CONFIG["cache_dir"], ignore_errors=True)
CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

def pipeline(ds, name, training):
    ds = ds.cache(str(CONFIG["cache_dir"] / name))     # DISK, not RAM
    if training:
        ds = ds.shuffle(CONFIG["shuffle_buffer"], seed=CONFIG["seed"],
                        reshuffle_each_iteration=True)
    return ds.batch(CONFIG["batch_size"]).prefetch(AUTOTUNE)

train_ds = pipeline(train_raw, "train", True)
val_ds   = pipeline(val_raw,   "val",   False)
test_ds  = pipeline(test_raw,  "test",  False)

# "Enhance quality" -- runs on the GPU as model layers, training only.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augmentation")

In [ ]:
import time
# Warm the disk cache once. Every epoch after this reads decoded tensors off SSD
# instead of re-decoding JPEGs, which is what keeps the GPU fed.
for name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    t0, n = time.time(), 0
    for xb, _ in ds:
        n += int(xb.shape[0])
    print(f"cached {name:>5}: {n:>6} images in {time.time() - t0:6.1f}s")

cache_mb = sum(f.stat().st_size for f in CONFIG["cache_dir"].glob("*")) / 1e6
print(f"\ncache on disk: {cache_mb:,.0f} MB (RAM usage stays flat)")

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_ds.take(1)))
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for i in range(6):
    axes[0, i].imshow(images[i].numpy().astype("uint8"))
    axes[0, i].set_title(CLASS_NAMES[labels[i]], fontsize=10)
    aug = data_augmentation(tf.expand_dims(images[i], 0), training=True)[0]
    axes[1, i].imshow(tf.cast(tf.clip_by_value(aug, 0, 255), tf.uint8).numpy())
    axes[1, i].set_title("augmented", fontsize=9, color="gray")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Top: original   |   Bottom: after augmentation", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Labels come from the file paths -- no need to decode a single image for this.
train_labels = np.array([CLASS_NAMES.index(Path(p).parent.name)
                         for p in train_raw.file_paths])
present = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=present, y=train_labels)
CLASS_WEIGHT = {int(c): float(w) for c, w in zip(present, weights)}

print(f"{'class':>8} {'n':>7} {'weight':>8}")
for i, c in enumerate(CLASS_NAMES):
    print(f"{c:>8} {int((train_labels == i).sum()):>7} {CLASS_WEIGHT.get(i, 0.0):>8.3f}")

## Step 4 — Model

EfficientNetV2B0 with a fresh 3-way head.

- **Mixed precision.** The T4 has float16 tensor cores; compute runs in float16 while
  weights stay float32. The softmax is forced back to float32 — in float16 it can
  overflow and produce `nan` loss.
- **Preprocessing is inside the model.** EfficientNetV2 carries its own normalization
  (`include_preprocessing=True`), so exported models take raw 0–255 images.
- **`base_model(x, training=False)`** keeps BatchNorm in inference mode. Letting BN
  statistics update while fine-tuning on a small dataset is the classic cause of
  "validation accuracy collapses the moment I unfreeze".

In [ ]:
from tensorflow.keras import layers, Model

if CONFIG["mixed_precision"] and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)

def build_model():
    base_model = BACKBONE_FN(IMG_SIZE + (3,))
    base_model.trainable = False

    inputs = layers.Input(shape=IMG_SIZE + (3,), name="image")
    x = data_augmentation(inputs)
    if PREPROCESS is not None:
        x = PREPROCESS(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="pool")(x)
    x = layers.Dropout(CONFIG["dropout"], name="dropout")(x)
    x = layers.Dense(NUM_CLASSES, name="logits")(x)
    # float32 softmax: float16 overflows here and gives nan loss
    outputs = layers.Activation("softmax", dtype="float32", name="predictions")(x)
    return Model(inputs, outputs, name="waste_classifier"), base_model

model, base_model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG["lr_head"]),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

print(f"backbone layers: {len(base_model.layers)}")
print(f"trainable params: {sum(int(tf.size(w)) for w in model.trainable_weights):,} "
      f"/ {sum(int(tf.size(w)) for w in model.weights):,}")

### Phase 1 — train the head

Only the new `Dense` layer learns. This plateaus quickly; that plateau is the ceiling
of frozen ImageNet features on this data.

In [ ]:
CKPT = str(CONFIG["export_dir"] / "best.keras")

history_head = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["epochs_head"],
    class_weight=CLASS_WEIGHT,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(CKPT, monitor="val_accuracy", mode="max",
                                           save_best_only=True, verbose=0),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4,
                                         restore_best_weights=True, verbose=1),
    ],
)
print(f"\nphase 1 best val accuracy: {max(history_head.history['val_accuracy']):.2%}")

### Phase 2 — fine-tune

Unfreeze the top 65% of the backbone (BatchNorm layers stay frozen) and train with
LR 1e-4 cosine-decayed to ~1e-6. `ReduceLROnPlateau` is deliberately absent: Keras
raises if you try to set the LR on an optimizer that owns a `LearningRateSchedule`.

In [ ]:
cut = int(len(base_model.layers) * CONFIG["unfreeze_frac"])
base_model.trainable = True
for layer in base_model.layers[:cut]:
    layer.trainable = False
for layer in base_model.layers:                      # BN frozen everywhere
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

steps_per_epoch = len(train_ds)
schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG["lr_finetune"],
    decay_steps=steps_per_epoch * CONFIG["epochs_finetune"],
    alpha=0.01)                                      # floor at 1% of the initial LR

model.compile(optimizer=tf.keras.optimizers.Adam(schedule),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

print(f"unfrozen from layer {cut}/{len(base_model.layers)}  |  "
      f"trainable params: {sum(int(tf.size(w)) for w in model.trainable_weights):,}")

history_ft = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["epochs_head"] + CONFIG["epochs_finetune"],
    initial_epoch=len(history_head.history["loss"]),
    class_weight=CLASS_WEIGHT,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(CKPT, monitor="val_accuracy", mode="max",
                                           save_best_only=True, verbose=0),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=8,
                                         restore_best_weights=True, verbose=1),
    ],
)

In [ ]:
def merge(k): return history_head.history.get(k, []) + history_ft.history.get(k, [])
acc, val_acc   = merge("accuracy"), merge("val_accuracy")
loss, val_loss = merge("loss"),     merge("val_loss")
split = len(history_head.history["loss"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, tr, va, name in [(a1, acc, val_acc, "Accuracy"), (a2, loss, val_loss, "Loss")]:
    ax.plot(tr, label="train", linewidth=2)
    ax.plot(va, label="validation", linewidth=2)
    ax.axvline(split - 0.5, color="crimson", ls="--", alpha=0.7, label="fine-tuning starts")
    ax.set_xlabel("epoch"); ax.set_title(name); ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"Best validation accuracy: {max(val_acc):.2%} (epoch {int(np.argmax(val_acc)) + 1})")
print(f"Gain from fine-tuning   : {max(val_acc) - max(val_acc[:split]):+.2%}")

## Step 5 — Evaluate

Test-time augmentation averages the prediction over the image plus its two flips.
Costs three forward passes, usually worth about a point, and needs no retraining.

**Read the per-class F1, not just overall accuracy.** `metal` and `wet` are the small
classes — a model can look fine overall and still miss them.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)

def predict_tta(m, ds):
    views = [lambda x: x, tf.image.flip_left_right, tf.image.flip_up_down]
    return np.mean([m.predict(ds.map(lambda x, y: (fn(x), y)), verbose=0)
                    for fn in views], axis=0)

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

y_prob_plain = model.predict(test_ds, verbose=0)
y_prob       = predict_tta(model, test_ds)

acc_plain = (np.argmax(y_prob_plain, 1) == y_true).mean()
y_pred    = np.argmax(y_prob, axis=1)
test_acc  = (y_pred == y_true).mean()

print(f"Test accuracy (plain): {acc_plain:.2%}")
print(f"Test accuracy (TTA)  : {test_acc:.2%}   ({test_acc - acc_plain:+.2%})\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.3f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES
    ).plot(ax=a1, cmap="Blues", colorbar=False, values_format="d")
a1.set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, normalize="true"),
                       display_labels=CLASS_NAMES
    ).plot(ax=a2, cmap="Blues", colorbar=False, values_format=".2f")
a2.set_title("Normalized by true class (recall on the diagonal)")
plt.tight_layout(); plt.show()

# The worst off-diagonal cell tells you which pair to attack next.
cm = confusion_matrix(y_true, y_pred, normalize="true").copy()
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(np.argmax(cm), cm.shape)
print(f"Biggest confusion: {CLASS_NAMES[i]} predicted as {CLASS_NAMES[j]} "
      f"({cm[i, j]:.0%} of all true {CLASS_NAMES[i]})")

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} misclassified out of {len(y_true)} ({len(wrong)/len(y_true):.1%})")
show = wrong[np.argsort(-y_prob[wrong].max(axis=1))][:8]

# Fetch only the few images we display -- concatenating the whole test set would
# put hundreds of MB of float32 pixels in RAM for eight thumbnails.
need, picked, idx = set(int(i) for i in show), {}, 0
for xb, _ in test_ds:
    for img in xb.numpy():
        if idx in need:
            picked[idx] = img
        idx += 1
    if len(picked) == len(need):
        break

if len(show):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, i in zip(axes.flat, show):
        ax.imshow(picked[int(i)].astype("uint8"))
        ax.set_title(f"true: {CLASS_NAMES[y_true[i]]}\n"
                     f"pred: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})",
                     fontsize=10, color="crimson")
        ax.axis("off")
    for ax in axes.flat[len(show):]:
        ax.axis("off")
    plt.suptitle("Most confident mistakes", y=1.0)
    plt.tight_layout(); plt.show()

## Step 6 — Export

The `.keras` file keeps the augmentation layers so training can resume. The TFLite
build strips them: TFLite has no native kernels for the random-transform ops and
conversion of the full model fails outright. The cell asserts both graphs agree
before converting.

Both take raw 0–255 RGB at the configured resolution. **Download the zip and send it
back — it holds everything the UI needs** (`.keras`, `.tflite`, `labels.json`).

In [ ]:
import json
EXPORT = CONFIG["export_dir"]

keras_path = EXPORT / "waste_classifier.keras"
model.save(keras_path)
print(f"Keras model : {keras_path.name}  ({keras_path.stat().st_size / 1e6:.1f} MB)")

inf_in = tf.keras.Input(shape=IMG_SIZE + (3,), name="image")
z = inf_in
if PREPROCESS is not None:
    z = PREPROCESS(z)
z = base_model(z, training=False)
for name in ("pool", "logits", "predictions"):
    z = model.get_layer(name)(z)
inference_model = tf.keras.Model(inf_in, z, name="waste_classifier_inference")

probe = np.random.randint(0, 256, (4,) + IMG_SIZE + (3,)).astype("float32")
# The two graphs share weights but differ slightly (one carries the augmentation
# layer), so float16 rounding lands differently. float16 holds ~3 decimal digits,
# so a legitimate gap of ~1e-3 is possible -- a 1e-4 tolerance would fail on a GPU
# run despite both models being correct.
tol = 2e-3 if tf.keras.mixed_precision.global_policy().compute_dtype == "float16" else 1e-5
np.testing.assert_allclose(model.predict(probe, verbose=0),
                           inference_model.predict(probe, verbose=0), atol=tol)
print(f"Inference graph matches the trained model (atol={tol:g})")

labels = {
    "class_names": CLASS_NAMES,
    "source_folders": SOURCE_FOLDERS,
    "excluded_folders": DROPPED_FOLDERS,
    "backbone": CONFIG["backbone"],
    "input_shape": [CONFIG["img_size"], CONFIG["img_size"], 3],
    "input_range": "0-255 RGB (preprocessing is inside the model)",
    "metrics": {
        "test_accuracy_tta": float(test_acc),
        "test_accuracy_plain": float(acc_plain),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "per_class_f1": {c: float(s) for c, s in
                         zip(CLASS_NAMES, f1_score(y_true, y_pred, average=None))},
    },
}
(EXPORT / "labels.json").write_text(json.dumps(labels, indent=2))
print("Labels      : labels.json")

tflite_path = EXPORT / "waste_classifier.tflite"
try:
    sm_dir = "/content/_saved_model"
    inference_model.export(sm_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_path.write_bytes(conv.convert())
    print(f"TFLite      : {tflite_path.name}  ({tflite_path.stat().st_size / 1e6:.1f} MB)")
except Exception as e:
    print(f"TFLite conversion skipped: {type(e).__name__}: {e}")

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/waste_classifier_export", "zip", EXPORT)
files.download(archive)

## Step 7 — Demo

In [ ]:
import gradio as gr
from PIL import Image

def predict(img):
    if img is None:
        return {}
    im = Image.fromarray(img).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(im, dtype="float32"), 0)
    probs = np.mean([inference_model.predict(v, verbose=0)[0] for v in
                     (arr, arr[:, :, ::-1], arr[:, ::-1])], axis=0)   # TTA
    return {c: float(p) for c, p in zip(CLASS_NAMES, probs)}

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Waste item"),
    outputs=gr.Label(num_top_classes=NUM_CLASSES, label="Bin"),
    title="Waste Segregation: wet / dry / metal",
    description=(f"{CONFIG['backbone']} fine-tuned on Garbage Classification. "
                 f"Test accuracy {test_acc:.1%} on clean single-item photos; "
                 "expect lower on cluttered real-world images."),
    flagging_mode="never",
).launch(share=True, debug=False)

## If the numbers fall short

Work the confusion matrix, not the hyperparameters. The cell in Step 5 prints the worst
confused pair — that one number tells you what to reach for:

1. **`wet` vs `dry` bleed into each other.** Look at the "most confident mistakes" —
   usually food-stained paper/cardboard or packaging with food on it. That's a genuine
   ambiguity in the labels, not a model bug.
2. **`metal` recall is low but no single confusor.** Only ~770 source images exist. Raise
   its weight, or add images (e.g. from TrashNet's `metal` folder) — and check the
   misclassified-metal thumbnails: foil and crushed cans often look like plastic.
3. **Train accuracy ≫ validation.** Overfitting: raise `dropout` toward 0.5, or
   strengthen augmentation.
4. **Both accuracies low and still climbing at the end.** Underfitting: raise
   `epochs_finetune`, or lower `unfreeze_frac` to 0.2.
5. **High test accuracy but poor results on your own photos.** Expected — the dataset is
   mostly clean single items on plain backgrounds. Collect 50–100 of your own photos per
   class (real bins, real lighting) and add them to the training folders; this beats any
   backbone change.

Bigger lever: `"backbone": "efficientnetv2b2"` with `"img_size": 260` (~2x training time).

**What not to do:** don't tune anything against the test set. Use validation accuracy
for every decision and touch the test set once, at the end.